# Supervised Learning for Airfoil Self Noise
**Abiell Yuman Atsal · 23625313 · AE5004**

Run all cells from top to bottom in a fresh Python 3 runtime. This notebook answers Tasks 1-6 and reproduces the report. It checks package versions, verifies the input file, and downloads one ZIP with cleaned data, exact model inputs, results and figures. No generated output is uploaded to GitHub or Drive.

The input comes directly from UCI because this dataset is not stored in the earlier homework repository. A local file or Colab upload is supported if the download is unavailable. Seed 42, 500 neural-network epochs, and the original candidate settings are retained. MSE values are reported in dB squared.


## Software configuration

In [ ]:
# Install the report versions before importing the scientific packages.
import sys, subprocess
from importlib.metadata import version, PackageNotFoundError
requirements = {'numpy': '2.5.0', 'pandas': '3.0.3', 'scipy': '1.18.0', 'matplotlib': '3.11.0', 'scikit-learn': '1.9.0', 'threadpoolctl': '3.6.0'}
missing=[]
for package,wanted in requirements.items():
    try:
        if version(package)!=wanted: missing.append(f'{package}=={wanted}')
    except PackageNotFoundError:
        missing.append(f'{package}=={wanted}')
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print({package:version(package) for package in requirements})


## Setup and dataset

In [ ]:
from pathlib import Path
import io, json, zipfile, urllib.request, platform, tempfile, hashlib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score
from threadpoolctl import threadpool_limits
threadpool_limits(limits=1)
SEED=42
EPOCHS=500
OUT=Path(tempfile.mkdtemp(prefix='supervised_regression_'))
FIG=OUT/'figures'; FIG.mkdir()
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
filename='airfoil_self_noise.dat'
expected_sha256='74c75fd71783f1e6b71f8a622b993dc592897a97cd689c5090a07147a1b097b3'
local=next((p for p in [Path('data')/filename,Path(filename)] if p.is_file()),None)
if local is not None:
    data=local.read_bytes()
else:
    try:
        url='https://archive.ics.uci.edu/static/public/291/airfoil+self+noise.zip'
        with urllib.request.urlopen(url,timeout=30) as response:
            raw=response.read()
        z=zipfile.ZipFile(io.BytesIO(raw))
        data=z.read(next(n for n in z.namelist() if n.endswith('.dat')))
    except Exception:
        try:
            from google.colab import files
        except ImportError as exc:
            raise RuntimeError('Place '+filename+' beside the notebook or in data/.') from exc
        print('Upload the original UCI file: '+filename)
        uploaded=files.upload()
        if filename not in uploaded: raise ValueError('Expected '+filename)
        data=uploaded[filename]
actual=hashlib.sha256(data).hexdigest()
if actual!=expected_sha256:
    raise ValueError('The input differs from the dataset used in the report.')
(OUT/filename).write_bytes(data)
print('Verified dataset SHA256:',actual)
cols=['Frequency (Hz)','Angle of attack (deg)','Chord length (m)','Velocity (m/s)','Displacement thickness (m)','Sound pressure level (dB)']
df=pd.read_csv(io.BytesIO(data),sep=r'\s+',header=None,names=cols)
assert df.shape==(1503,6) and df.isna().sum().sum()==0
print('Rows:',len(df),'Missing values:',df.isna().sum().sum(),'Duplicate rows:',df.duplicated().sum())
stats=df.describe().T[['mean','std','min','50%','max']]
stats.to_csv(OUT/'statistics.csv'); print(stats.to_string())
fig,axs=plt.subplots(2,3,figsize=(10,5.3))
for ax,c in zip(axs.flat,cols):
 ax.hist(df[c],bins=25,color='#28758f',edgecolor='white'); ax.set(xlabel=c,ylabel='Count')
fig.tight_layout(); fig.savefig(FIG/'distributions.png',dpi=180); plt.close(fig)
X=df.iloc[:,:5].to_numpy(); y=df.iloc[:,5].to_numpy()
idx=np.arange(len(y))
itr,itemp=train_test_split(idx,test_size=.30,random_state=SEED)
iva,ite=train_test_split(itemp,test_size=.50,random_state=SEED)
idev=np.concatenate([itr,iva])
assert not(set(itr)&set(iva) or set(itr)&set(ite) or set(iva)&set(ite))
print('Train, validation, test:',len(itr),len(iva),len(ite))
pd.DataFrame({'row':idx,'split':['train' if i in itr else 'validation' if i in iva else 'test' for i in idx]}).to_csv(OUT/'split.csv',index=False)



In [ ]:
from IPython.display import display, Image
display(Image(filename=str(FIG/'distributions.png')))

## Tasks 2-4: original validation set

In [ ]:
degrees=[1,2,3]; neighbors=[1,3,5,9,15,25]; architectures=[(32,32),(32,),(32,32,32),(64,64)]
def label(a): return '-'.join(map(str,a))
def classical(family,parameter):
 if family=='Polynomial': return make_pipeline(StandardScaler(),PolynomialFeatures(parameter,include_bias=False),LinearRegression())
 return make_pipeline(StandardScaler(),KNeighborsRegressor(n_neighbors=parameter))
def fit_nn(train_idx,eval_idx,arch,record=False):
 sx=StandardScaler().fit(X[train_idx]); sy=StandardScaler().fit(y[train_idx,None])
 xt=sx.transform(X[train_idx]); yt=sy.transform(y[train_idx,None]).ravel()
 xe=sx.transform(X[eval_idx]) if eval_idx is not None else None
 net=MLPRegressor(hidden_layer_sizes=arch,activation='relu',solver='adam',alpha=0.0,batch_size=32,learning_rate_init=.001,random_state=SEED,shuffle=True)
 history=[]
 for epoch in range(EPOCHS):
  net.partial_fit(xt,yt)
  if record:
   pt=sy.inverse_transform(net.predict(xt)[:,None]).ravel()
   pe=sy.inverse_transform(net.predict(xe)[:,None]).ravel()
   history.append([mean_squared_error(y[train_idx],pt),mean_squared_error(y[eval_idx],pe)])
 def predict(xx): return sy.inverse_transform(net.predict(sx.transform(xx))[:,None]).ravel()
 return predict,history
rows=[]; histories={}
for family,params in [('Polynomial',degrees),('KNN',neighbors)]:
 for p in params:
  m=classical(family,p).fit(X[itr],y[itr])
  rows.append({'Family':family,'Setting':str(p),'Train MSE':mean_squared_error(y[itr],m.predict(X[itr])),'Validation MSE':mean_squared_error(y[iva],m.predict(X[iva]))})
for a in architectures:
 pred,h=fit_nn(itr,iva,a,True); histories[label(a)]=h
 rows.append({'Family':'Neural network','Setting':label(a),'Train MSE':mean_squared_error(y[itr],pred(X[itr])),'Validation MSE':mean_squared_error(y[iva],pred(X[iva]))})
 print('Finished validation NN',a,flush=True)
validation=pd.DataFrame(rows); validation.to_csv(OUT/'validation.csv',index=False); print(validation.to_string(index=False))
kn=validation[validation.Family=='KNN']
fig,ax=plt.subplots(figsize=(8,3))
ax.plot(neighbors,kn['Train MSE'],'o-',label='Training'); ax.plot(neighbors,kn['Validation MSE'],'s-',label='Validation'); ax.set(xlabel='Number of neighbors, K',ylabel='MSE (dB squared)'); ax.legend(); fig.tight_layout(); fig.savefig(FIG/'knn.png',dpi=180); plt.close(fig)
fig,axs=plt.subplots(2,2,figsize=(9,5.3),sharex=True,sharey=True)
for ax,(name,h) in zip(axs.flat,histories.items()):
 h=np.array(h); ax.plot(np.arange(1,EPOCHS+1),h[:,0],label='Training'); ax.plot(np.arange(1,EPOCHS+1),h[:,1],label='Validation'); ax.set(title='Hidden layers: '+name,xlabel='Epoch',ylabel='MSE (dB squared)',yscale='log'); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG/'nn_curves.png',dpi=180); plt.close(fig)



In [ ]:
from IPython.display import display, Image
display(validation.round(4))
display(Image(filename=str(FIG/'knn.png')))
display(Image(filename=str(FIG/'nn_curves.png')))

## Task 5: five-fold cross-validation; test data remain unused

In [ ]:
folds=list(KFold(5,shuffle=True,random_state=SEED).split(idev))
cvrows=[]
for family,params in [('Polynomial',degrees),('KNN',neighbors),('Neural network',architectures)]:
 for p in params:
  scores=[]
  for it,iv in folds:
   tr,va=idev[it],idev[iv]
   if family=='Neural network': pred,_=fit_nn(tr,va,p)
   else: pred=classical(family,p).fit(X[tr],y[tr]).predict
   scores.append(mean_squared_error(y[va],pred(X[va])))
  row={'Family':family,'Setting':label(p) if family=='Neural network' else str(p),'Mean CV MSE':np.mean(scores),'SD CV MSE':np.std(scores,ddof=0)}
  row.update({f'Fold {j+1}':v for j,v in enumerate(scores)}); cvrows.append(row)
  print('CV:',family,p,round(np.mean(scores),4),flush=True)
cv=pd.DataFrame(cvrows); cv.to_csv(OUT/'cross_validation.csv',index=False)



In [ ]:
from IPython.display import display, Image
display(cv.round(4))

## Task 6: freeze selection, refit on development data, then test once

In [ ]:
best=cv.loc[cv.groupby('Family')['Mean CV MSE'].idxmin()]
results=[]; predictions={}
for _,row in best.iterrows():
 family,p=row['Family'],row['Setting']
 if family=='Neural network': pred,_=fit_nn(idev,None,tuple(map(int,p.split('-'))))
 else: pred=classical(family,int(p)).fit(X[idev],y[idev]).predict
 yp=pred(X[ite]); predictions[family]=yp
 mse=mean_squared_error(y[ite],yp)
 results.append({'Model':family,'Setting':p,'Test MSE':mse,'Test RMSE':np.sqrt(mse),'Test R2':r2_score(y[ite],yp)})
final=pd.DataFrame(results); final.to_csv(OUT/'test_results.csv',index=False); print(final.to_string(index=False))
fig,axs=plt.subplots(1,3,figsize=(10,3.4),sharex=True,sharey=True)
for ax,(name,yp) in zip(axs,predictions.items()):
 ax.scatter(y[ite],yp,s=12,alpha=.65,color='#28758f'); ax.plot([100,145],[100,145],'k--',lw=1); ax.set(title=name,xlabel='Observed level (dB)',xlim=(100,145),ylim=(100,145))
axs[0].set_ylabel('Predicted level (dB)'); fig.tight_layout(); fig.savefig(FIG/'predictions.png',dpi=180); plt.close(fig)
(OUT/'metadata.json').write_text(json.dumps({'seed':SEED,'epochs':EPOCHS,'split':[len(itr),len(iva),len(ite)],'python':platform.python_version(),'sklearn':sklearn.__version__,'numpy':np.__version__,'duplicates':int(df.duplicated().sum())},indent=2))


In [ ]:
from IPython.display import display, Image
display(final.round(4))
display(Image(filename=str(FIG/'predictions.png')))

## Download cleaned data and results
These cells package all results into one ZIP. They do not require repository write access.

In [ ]:
# Preserve row identities, cleaning audit and exact scaled model inputs.
cleaned=df.copy()
cleaned.insert(0,'source_row_0based',np.arange(len(df)))
cleaned.to_csv(OUT/'cleaned_airfoil.csv',index=False)
cleaned.iloc[:0].assign(exclusion_reason=pd.Series(dtype=str)).to_csv(OUT/'excluded_rows.csv',index=False)
scaler_rows=[]
def export_inputs(name,fit_rows,eval_groups):
    sx=StandardScaler().fit(X[fit_rows])
    sy=StandardScaler().fit(y[fit_rows,None])
    for feature,mean,scale in zip(cols[:5],sx.mean_,sx.scale_):
        scaler_rows.append({'fit_context':name,'variable':feature,'mean':mean,'scale_ddof0':scale})
    scaler_rows.append({'fit_context':name,'variable':cols[5],'mean':sy.mean_[0],'scale_ddof0':sy.scale_[0]})
    for group,indices in eval_groups.items():
        frame=pd.DataFrame(sx.transform(X[indices]),columns=cols[:5])
        frame.insert(0,'source_row_0based',indices)
        frame['target_dB']=y[indices]
        frame['target_standardized_for_NN']=sy.transform(y[indices,None]).ravel()
        frame.to_csv(OUT/f'{name}_{group}_inputs.csv',index=False)
export_inputs('original',itr,{'train':itr,'validation':iva,'test':ite})
export_inputs('final',idev,{'development':idev,'test':ite})
fold_labels=[]
for fold,(tr,va) in enumerate(folds,1):
    tr,va=idev[tr],idev[va]
    export_inputs(f'cv_fold_{fold}',tr,{'train':tr,'validation':va})
    fold_labels.extend({'fold':fold,'source_row_0based':int(i),'role':role} for role,indices in [('train',tr),('validation',va)] for i in indices)
pd.DataFrame(scaler_rows).to_csv(OUT/'standardization_parameters.csv',index=False)
pd.DataFrame(fold_labels).to_csv(OUT/'cv_fold_assignments.csv',index=False)
for name,history in histories.items():
    pd.DataFrame(history,columns=['Train MSE','Validation MSE']).assign(epoch=np.arange(1,EPOCHS+1)).to_csv(OUT/f'NN_{name}_learning_curve.csv',index=False)
prediction_table=pd.DataFrame({'source_row_0based':ite,'observed_dB':y[ite],**predictions})
prediction_table.to_csv(OUT/'test_predictions.csv',index=False)
best.to_csv(OUT/'selected_hyperparameters.csv',index=False)
audit={'input_sha256':actual,'original_rows':len(df),'cleaned_rows':len(cleaned),
       'excluded_rows':0,'missing_values':int(df.isna().sum().sum()),'duplicate_rows':int(df.duplicated().sum()),
       'cleaning_rule':'All input rows are complete and unique. No rows removed or imputed.',
       'split_sizes':{'train':len(itr),'validation':len(iva),'test':len(ite)},
       'seed':SEED,'epochs':EPOCHS,'cv_folds':5,'cv_sd_ddof':0,
       'model_selection':'Minimum mean CV MSE within each model family. Test used only after selection.',
       'nn_settings':{'activation':'relu','optimizer':'adam','learning_rate':.001,'batch_size':32,'alpha':0.0},
       'source_row_0based':'Original data-row position starting at zero.'}
(OUT/'data_audit.json').write_text(json.dumps(audit,indent=2))
README="""Supervised regression results
cleaned_airfoil.csv: all 1503 observations, their original units and source-row identities.
excluded_rows.csv: header-only because there are no excluded rows.
airfoil_self_noise.dat: verified unchanged input file.
split.csv: the original train/validation/test row split.
original_*_inputs.csv: model inputs standardized using only the original training rows.
final_*_inputs.csv: final model inputs standardized using all development rows.
cv_fold_*_inputs.csv: exact standardized inputs for each fold, using that fold's training scaler.
Standardized target columns apply to neural networks only. Polynomial and KNN targets stay in dB.
Polynomial features are constructed from standardized inputs, as shown in the notebook.
standardization_parameters.csv: fitted input and target means and population standard deviations.
cv_fold_assignments.csv: row membership for the five development folds.
validation.csv, cross_validation.csv, selected_hyperparameters.csv: complete model-selection results.
test_results.csv and test_predictions.csv: final metrics and per-row predictions.
NN_*_learning_curve.csv: original train and validation MSE for every epoch.
statistics.csv, metadata.json, data_audit.json, software_versions.json: statistics and reproducibility records.
figures/: all figures displayed in the notebook.
Test targets are exported for checking; they are never used as model inputs or for model selection.
All generated files are local to this session. Nothing is uploaded to GitHub or Drive.
"""


In [ ]:
import shutil
from importlib.metadata import version
(OUT/'software_versions.json').write_text(json.dumps({p:version(p) for p in requirements},indent=2))
(OUT/'README_outputs.txt').write_text(README)
archive=Path(shutil.make_archive(str(OUT.parent/(OUT.name+'_Results')),'zip',root_dir=OUT))
print('Created:',archive)
try:
    from google.colab import files
except ImportError:
    print('Local ZIP path:',archive)
else:
    files.download(str(archive))


## Conclusion
The 64-64 neural network gave the lowest test MSE of 2.2623 and RMSE of 1.5041 dB in the report run. Degree 3 was best for polynomial regression, while K = 1 was best for KNN. This comparison uses a random row split; new experimental conditions need a separate grouped evaluation.

## Source
Brooks, Pope and Marcolini (1989), Airfoil Self-Noise, UCI Machine Learning Repository. https://doi.org/10.24432/C5VW2C